# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Bisma6405/flyrank_internship_ml/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [1]:
# Setup: load the starter data. Works from the repo folder or from Colab.
import os
import numpy as np
import pandas as pd

CANDIDATES = [
    "data/raw/content_refresh_anonymized.csv",
    "../../data/raw/content_refresh_anonymized.csv",
    "../data/raw/content_refresh_anonymized.csv",
]
path = next((p for p in CANDIDATES if os.path.exists(p)), None)
if path is None:  # Colab opened the notebook alone: read the file from the repo
    path = ("https://raw.githubusercontent.com/Bisma6405/flyrank_internship_ml/"
            "main/data/raw/content_refresh_anonymized.csv")
df = pd.read_csv(path)
print("Loaded:", df.shape)

Loaded: (30000, 44)


## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

**Classification, used as a ranking.**

For each page, I predict a yes/no answer: *"Is this page losing clicks?"* Then I sort pages by the model's score, highest first. The top of that list is the "look at these first" queue.

Why classification: the answer is a clear yes or no, and I can score it against a real label.
Why not clustering: I have a label, so I don't need to guess the groups.
Why ranking matters: a team can only review a few pages, so the order of the list is what counts.

In [2]:
print("Rows, columns:", df.shape)
print("Columns:", df.columns.tolist())

Rows, columns: (30000, 44)
Columns: ['content_id', 'client_id', 'search_volume', 'competition', 'competition_level', 'cpc', 'content_type', 'main_intent', 'word_count', 'char_count', 'provider_used', 'model_used', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days', 'age_tier', 'age_tier_order', 'days_since_last_update', 'freshness_tier', 'word_count_tier', 'char_count_tier', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'impression_tier', 'position_tier', 'trend_direction', 'trend_pct']


## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*


**What I predict:** `losing_clicks` = 1 if a page's clicks in the last 30 days fell by 30% or more compared with the 30 days before. Otherwise 0.

**Where the label comes from:** a **defined rule**, not a direct outcome, so it is a **proxy**. It stands in for "this page may need a refresh." A click drop can also come from season, or from a search change. So the label is directional, not proof.

**Who is in the slice:** only pages with at least 5 clicks in the earlier 30 days. With fewer clicks, a "drop" is just noise.

**Columns kept out of the inputs:** the columns that build the label (`clicks_last_30d`, `clicks_prev_30d`) and the ones that are almost the same thing (`trend_pct`, `trend_direction`, `*_30d`). If they stay in, the model just reads the answer back. I will check leakage in more detail in week 3.


In [4]:
MIN_PREV_CLICKS = 5     # need enough clicks to see a real drop
DROP = 0.30             # 30% fall or more = "losing clicks"

d = df[df["clicks_prev_30d"] >= MIN_PREV_CLICKS].copy()
d["losing_clicks"] = (d["clicks_last_30d"] <= (1 - DROP) * d["clicks_prev_30d"]).astype(int)

print("Pages in slice:", len(d), "of", len(df))
print(d["losing_clicks"].value_counts())
print("Share losing clicks:", round(d["losing_clicks"].mean(), 3))


Pages in slice: 5019 of 30000
losing_clicks
0    3030
1    1989
Name: count, dtype: int64
Share losing clicks: 0.396


## 3. Success metric

*One metric you can defend. What number means 'good'?*

**Precision@50.** Of the 50 pages the model ranks highest, what share really are losing clicks?

Why this one: the team reviews a short list, not every page. Precision@50 asks the same question the team asks: *is the top of my list worth their time?*

**What "good" means:** it must beat two things on pages the model has never seen:
1. the **base rate** (pick 50 pages at random), and
2. a **simple rule** (for example, "oldest update first").

I test on **different clients** than I train on. That is stricter, because the model has not seen those sites.

In [5]:
from sklearn.model_selection import GroupShuffleSplit
from sklearn.ensemble import RandomForestClassifier

# Inputs: page and traffic signals only. No label columns, no *_30d, no trend columns.
FEATURES = ["search_volume", "competition", "cpc", "word_count", "impressions_90d",
            "clicks_90d", "days_with_impressions", "content_age_days",
            "days_since_last_update", "ctr", "avg_position", "engagement_rate",
            "scroll_rate", "ai_traffic_pct"]
X = d[FEATURES].fillna(-1)
y = d["losing_clicks"]

# Hold out whole clients so the test pages come from sites the model never saw
train_i, test_i = next(GroupShuffleSplit(n_splits=1, test_size=0.3, random_state=1)
                       .split(X, y, groups=d["client_id"]))
X_tr, X_te, y_tr, y_te = X.iloc[train_i], X.iloc[test_i], y.iloc[train_i], y.iloc[test_i]

def precision_at_k(y_true, score, k=50):
    top = pd.Series(np.asarray(score), index=y_true.index).nlargest(k).index
    return y_true.loc[top].mean()

model = RandomForestClassifier(n_estimators=200, min_samples_leaf=20, n_jobs=-1, random_state=0)
model.fit(X_tr, y_tr)
model_score = model.predict_proba(X_te)[:, 1]

print("Test pages:", len(y_te), "| test clients:", d.iloc[test_i]["client_id"].nunique())
print("Base rate (random 50):    ", round(y_te.mean(), 3))
print("Model Precision@50:       ", round(precision_at_k(y_te, model_score), 3))

Test pages: 569 | test clients: 8
Base rate (random 50):     0.362
Model Precision@50:        0.7


## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*


**One row = one page** (one `content_id`). Each page belongs to one `client_id`.

The table below shows the columns I use. The last column, `losing_clicks`, is my target. Each row has the page's traffic signals, and the target says if its clicks fell.

In [6]:
print("Each row is one page. Unique pages:", d["content_id"].is_unique)
show = ["content_id", "avg_position", "impressions_90d",
        "days_since_last_update", "content_age_days",
        "clicks_prev_30d", "clicks_last_30d", "losing_clicks"]
d[show].head(8)

Each row is one page. Unique pages: True


,content_id,avg_position,impressions_90d,days_since_last_update,content_age_days,clicks_prev_30d,clicks_last_30d,losing_clicks
0,content_304f48230142,10.6,3803,20,187,13,2,1
3,content_331d6c4de07b,6.2,11751,22,463,17,22,0
8,content_5e6c160719bc,46.0,32574,20,90,8,9,0
10,content_d8ee6cc6d642,2.2,20919,104,329,117,112,0
12,content_42fb2cad9ecf,5.6,7228,104,124,29,92,0
16,content_78bd1d4a1d4d,8.9,13848,104,307,5,9,0
24,content_0e23e310d404,4.1,29541,104,257,31,26,0
38,content_dbe82879a406,33.3,26457,104,223,7,11,0


## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

A fixed rule looks at one signal with one cutoff, like "update the oldest pages first" or "look at pages with a weak position." But clicks fall for many mixed reasons: position, how much traffic the page has, its age, and how it is written. These signals act together, and the cutoffs are different for each page.

Below I score two simple rules on the same test pages, and compare them with the model. Same pages, same metric.

**Content action:** the model's ranked list tells the content team which pages to review first. They then decide what to change. It is decision-support, not an order.

In [7]:
te = d.iloc[test_i]

rule_age = te["days_since_last_update"]      # rule 1: oldest update first
rule_pos = te["avg_position"]                # rule 2: weakest position first

results = pd.DataFrame({
    "method": ["Random pick (base rate)", "Rule: oldest update first",
               "Rule: weakest position first", "Model (random forest)"],
    "Precision@50": [y_te.mean(),
                     precision_at_k(y_te, rule_age),
                     precision_at_k(y_te, rule_pos),
                     precision_at_k(y_te, model_score)],
}).round(3)
display(results)

# The model list is the "fix this first" queue:
queue = te.assign(score=model_score).nlargest(10, "score")[
    ["content_id", "avg_position", "days_since_last_update", "impressions_90d", "score"]]
print("Top 10 of the 'fix this first' queue:")
display(queue.round(3))

,method,Precision@50
0,Random pick (base rate),0.362
1,Rule: oldest update first,0.420
2,Rule: weakest position first,0.380
3,Model (random forest),0.700


Top 10 of the 'fix this first' queue:


,content_id,avg_position,days_since_last_update,impressions_90d,score
17973,content_c897409546b8,9.1,20,1717,0.884
23398,content_e4909f5dc6f6,6.0,20,1200,0.869
14021,content_0be397916dac,5.8,20,986,0.867
8167,content_08975daa4db2,4.9,20,1861,0.866
27877,content_195216abb0e4,8.9,20,1815,0.853
21644,content_29ab22115ce9,7.6,20,822,0.851
19170,content_9420383ea370,8.1,20,454,0.846
18740,content_32d99307d773,7.7,104,4433,0.845
13384,content_d9dfddab4875,8.7,20,1552,0.845
14691,content_5af5e83de3cf,6.6,8,2883,0.836


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.